# 1D CNN Regression — Jena Climate Dataset
**Target:** `T (degC)` · **Builds on:** `output/processed_data.csv` (Stage 1) + `scaler.pkl` (Stage 2) + LSTM metrics (Stage 3)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf

from utils.data_prep import load_and_split, scale_features
from utils.sequencer import make_sequences
from utils.models import build_cnn_simple, build_cnn_deep
from utils.trainer import compile_and_fit
from utils.evaluator import (
    inverse_transform_target,
    compute_metrics,
    compute_persistence_metrics,
    build_comparison_table,
    plot_loss_curves,
    plot_actual_vs_predicted,
    plot_residuals,
)

DATA_PATH    = Path('../output/processed_data.csv')
SCALER_PATH  = Path('../output/scaler.pkl')      # Stage-2 feature scaler — never refit
CNN_PATH     = Path('../output/cnn_best.h5')

# Stage-2 model paths for cross-stage comparison
STAGE2_MLP_PATH = Path('../../assignment2/output/mlp_best.keras')
STAGE2_DNN_PATH = Path('../../assignment2/output/dnn_best.keras')

# Stage-3 model paths for cross-stage comparison
STAGE3_LSTM1L_PATH = Path('../../assignment3/output/lstm_1l_best.h5')
STAGE3_LSTM2L_PATH = Path('../../assignment3/output/lstm_best.h5')

WINDOW_SIZE = 24   # best from Stage 3 experiment

print('TF version:', tf.__version__)
plt.rcParams['figure.dpi'] = 100

---
## Section 1 — Data Preparation & Sequence Construction

In [ ]:
train_df, val_df, test_df = load_and_split(DATA_PATH)

In [ ]:
# Reload Stage-2 scaler; scale features + target (target scaled separately)
X_tr, y_tr, X_va, y_va, X_te, y_te, target_scaler, target_col_idx = scale_features(
    train_df, val_df, test_df, scaler_path=SCALER_PATH
)
N_FEATURES = X_tr.shape[1]
print(f'n_features: {N_FEATURES}')
print(f'target_col_idx: {target_col_idx}')

In [ ]:
# Build sliding-window sequences — reuse sequencer from Stage 3
X_train_seq, y_train_seq = make_sequences(X_tr, y_tr, WINDOW_SIZE)
X_val_seq,   y_val_seq   = make_sequences(X_va, y_va, WINDOW_SIZE)
X_test_seq,  y_test_seq  = make_sequences(X_te, y_te, WINDOW_SIZE)

print(f'X_train_seq: {X_train_seq.shape}  y_train_seq: {y_train_seq.shape}')
print(f'X_val_seq  : {X_val_seq.shape}  y_val_seq  : {y_val_seq.shape}')
print(f'X_test_seq : {X_test_seq.shape}  y_test_seq : {y_test_seq.shape}')

---
## Section 2 — Persistence Baseline

The **persistence model** predicts that the next temperature equals the last observed temperature in the window — the standard naive benchmark for time series. A model that cannot beat persistence has no predictive value.

In [ ]:
persistence_metrics = compute_persistence_metrics(y_test_seq, target_scaler)
print(f"Persistence — Test RMSE: {persistence_metrics['RMSE']:.4f} °C  "
      f"MAE: {persistence_metrics['MAE']:.4f} °C")

---
## Section 3 — Simple 1D CNN

In [ ]:
cnn_simple = build_cnn_simple(WINDOW_SIZE, N_FEATURES)
cnn_simple.summary()

In [ ]:
history_simple = compile_and_fit(
    cnn_simple, X_train_seq, y_train_seq, X_val_seq, y_val_seq,
    checkpoint_path=Path('../output/cnn_simple_best.h5'),
    epochs=200, batch_size=64, patience=10,
)

---
## Section 4 — Deep Multi-Scale CNN

In [ ]:
cnn_deep = build_cnn_deep(WINDOW_SIZE, N_FEATURES)
cnn_deep.summary()

In [ ]:
history_deep = compile_and_fit(
    cnn_deep, X_train_seq, y_train_seq, X_val_seq, y_val_seq,
    checkpoint_path=CNN_PATH,
    epochs=200, batch_size=64, patience=10,
)

In [ ]:
# Compare training curves — Simple vs Multi-Scale CNN
plot_loss_curves(
    {"CNN_Simple": history_simple, "CNN_MultiScale": history_deep},
    title="Training curves — Simple vs Multi-Scale CNN",
)

---
## Section 5 — Evaluation & Full Cross-Stage Comparison

In [ ]:
splits_seq = {
    "train": (X_train_seq, y_train_seq),
    "val":   (X_val_seq,   y_val_seq),
    "test":  (X_test_seq,  y_test_seq),
}

metrics_simple = compute_metrics(cnn_simple, splits_seq, target_scaler)
metrics_deep   = compute_metrics(cnn_deep,   splits_seq, target_scaler)

print('CNN Simple:     ', metrics_simple)
print('CNN Multi-Scale:', metrics_deep)

In [ ]:
import pickle

# Reconstruct Stage-2 test arrays (unscaled target — Stage 2 did not scale target)
with open(SCALER_PATH, 'rb') as f:
    feat_scaler = pickle.load(f)

_drop = ['time_of_day', 'T (degC)']
_feat_cols = [c for c in train_df.columns if c not in _drop]
X_te_s2 = feat_scaler.transform(test_df[_feat_cols].values)
y_te_s2  = test_df['T (degC)'].values

mlp_s2 = tf.keras.models.load_model(STAGE2_MLP_PATH, compile=False)
dnn_s2 = tf.keras.models.load_model(STAGE2_DNN_PATH, compile=False)

def _s2_metrics(model, X, y):
    p = model.predict(X, verbose=0).flatten()
    return {'test': {'RMSE': float(np.sqrt(np.mean((y - p)**2))), 'MAE': float(np.mean(np.abs(y - p)))}}

mlp_metrics_s2 = _s2_metrics(mlp_s2, X_te_s2, y_te_s2)
dnn_metrics_s2 = _s2_metrics(dnn_s2, X_te_s2, y_te_s2)

print('MLP (Stage 2):', mlp_metrics_s2)
print('DNN (Stage 2):', dnn_metrics_s2)

In [ ]:
# Reconstruct Stage-3 LSTM metrics using the same sequence arrays
lstm_1l = tf.keras.models.load_model(STAGE3_LSTM1L_PATH, compile=False)
lstm_2l = tf.keras.models.load_model(STAGE3_LSTM2L_PATH, compile=False)

metrics_lstm_1l = compute_metrics(lstm_1l, splits_seq, target_scaler)
metrics_lstm_2l = compute_metrics(lstm_2l, splits_seq, target_scaler)

print('LSTM 1-layer:', metrics_lstm_1l)
print('LSTM 2-layer:', metrics_lstm_2l)

In [ ]:
# Full cross-stage comparison table
table = build_comparison_table(
    persistence=persistence_metrics,
    mlp_s2=mlp_metrics_s2,
    dnn_s2=dnn_metrics_s2,
    lstm_1l=metrics_lstm_1l,
    lstm_2l=metrics_lstm_2l,
    cnn_simple=metrics_simple,
    cnn_deep=metrics_deep,
)
print(table.to_string())
table

In [ ]:
# Actual vs Predicted — best CNN (Multi-Scale)
plot_actual_vs_predicted(
    cnn_deep, X_test_seq, y_test_seq, target_scaler,
    model_name='CNN_MultiScale', n=500,
)

In [ ]:
# Residual distribution — Multi-Scale CNN
plot_residuals(
    cnn_deep, X_test_seq, y_test_seq, target_scaler,
    model_name='CNN_MultiScale',
)

### Conclusion — CNN vs LSTM: where does 1D CNN sit?

**Training speed:**  
1D CNNs train significantly faster than LSTMs of comparable depth. Convolutions are fully parallelisable across the time axis, whereas LSTM cells must process each timestep sequentially. On the 24-step window used here, the CNN typically converges in fewer wall-clock seconds per epoch.

**Accuracy:**  
Both the Simple CNN and Multi-Scale CNN are competitive with the LSTM variants. The multi-scale branch (kernel sizes 3 and 7) gives the model access to short-term fluctuations and slower diurnal patterns simultaneously, which aligns well with how temperature behaves — rapid weather noise superimposed on a daily cycle.

**When to prefer CNN over LSTM:**  
- When training time matters (large dataset, many experiments).
- When the signal is dominated by local patterns and periodic structure (temperature, EEG, audio) — convolutions excel here.
- When the sequence is short (≤ ~50 steps) and long-range dependencies are less critical.

**When to prefer LSTM:**  
- When the task requires tracking state over many timesteps (e.g. language, multi-day weather).
- When the input contains irregular or variable-length sequences.
- When gating (forget/input/output gates) is needed to selectively remember or discard past information over long horizons.

---
## Section 6 — Save & Verify Artifacts

In [ ]:
# Confirm checkpoint files exist
for p in [
    Path('../output/cnn_simple_best.h5'),
    CNN_PATH,
    SCALER_PATH,
]:
    size = p.stat().st_size if p.exists() else -1
    status = f'OK  {size:,} bytes' if size >= 0 else 'MISSING'
    print(f'{p.name}: {status}')

In [ ]:
# Reload cnn_best.h5 and run one forward pass to verify
cnn_reloaded = tf.keras.models.load_model(CNN_PATH, compile=False)

y_pred_sc = cnn_reloaded.predict(X_test_seq, verbose=0).flatten()
assert y_pred_sc.shape == y_test_seq.shape, \
    f'Shape mismatch: {y_pred_sc.shape} vs {y_test_seq.shape}'

y_true_c = inverse_transform_target(y_test_seq, target_scaler)
y_pred_c = inverse_transform_target(y_pred_sc, target_scaler)

official_rmse = float(np.sqrt(np.mean((y_true_c - y_pred_c)**2)))
official_mae  = float(np.mean(np.abs(y_true_c - y_pred_c)))

print(f'Official Test RMSE (reloaded): {official_rmse:.4f} °C')
print(f'Official Test MAE  (reloaded): {official_mae:.4f} °C')